In [1]:
import pandas as pd
from sklearn.neighbors import NearestNeighbors
import joblib
import numpy as np
import time
import os
from pathlib import Path
import json
import math
import requests
import scipy.sparse as sp

In [2]:
svd = joblib.load('svd_model.joblib')
dense_user_vectors = np.load('svd_vectors.npy')
df_normalized = sp.load_npz('df_norm_sparse.npz')
with open('anime_names.json', 'r', encoding='utf-8') as f:
    anime_names = json.load(f)

In [3]:
conf_path = Path("../tauri.conf.json")

with open(conf_path, 'r') as f:
    config = json.load(f)

In [4]:
app_data_path = Path(os.getenv('APPDATA') or Path.home() / ".local/share")

watchlist_path = app_data_path / config['identifier'] / 'watchlist.json'
notwatchlist_path = app_data_path / config['identifier'] / 'notwatchlist.json'
cache_path = app_data_path / config['identifier'] / 'cache.json'

In [5]:
user_data = pd.read_json(watchlist_path)
not_wanted = pd.read_json(notwatchlist_path)
cached_data = pd.read_json(cache_path)

user_data = user_data.to_dict(orient='records')
not_wanted = not_wanted.to_dict(orient='records')
cached_data = cached_data.to_dict(orient='records')
not_wanted_set = set([int(elem['mal_id']) for elem in not_wanted if pd.notna(elem['mal_id'])] + [int(elem['mal_id']) for elem in cached_data if pd.notna(elem['mal_id'])])
print(not_wanted_set)

{1, 5, 28683, 21, 40982, 53273, 20509, 61469, 40991, 36896, 32801, 28701, 38958, 36914, 36915, 8247, 36923, 38972, 47164, 4155, 2107, 57413, 59465, 2124, 36946, 16468, 36949, 32863, 36962, 41059, 59493, 20583, 30831, 56196, 16498, 41075, 32887, 121, 24703, 22661, 136, 137, 138, 139, 49318, 22695, 32935, 32937, 51367, 22699, 57519, 59571, 59596, 51410, 57555, 57556, 57557, 28891, 37087, 223, 225, 37095, 61674, 49387, 57584, 18679, 63736, 22777, 57592, 24833, 35073, 59654, 269, 6421, 12565, 33047, 53529, 33050, 33049, 35110, 35111, 35118, 49458, 57658, 43325, 63806, 39239, 53580, 51535, 31056, 63824, 2386, 2385, 53588, 43349, 63830, 41306, 57691, 55644, 55647, 356, 59765, 33142, 41341, 14719, 33155, 33156, 20871, 12685, 37264, 55701, 59801, 37278, 24991, 49569, 31138, 18851, 20899, 53672, 31145, 430, 59822, 25011, 59833, 12729, 2490, 35262, 18881, 59841, 61892, 459, 460, 461, 462, 463, 464, 465, 466, 2520, 49627, 49633, 37347, 37348, 14829, 37362, 4596, 14837, 502, 61942, 35321, 41467, 4

In [6]:
t0 = time.time()

In [7]:
user_ratings = {
    #f"{entry['mal_id']}": f"{entry['score']}"
    f"{entry['mal_id']}_{entry['title']}": entry['score']
    for entry in user_data
    if entry.get('score') is not None and not (isinstance(entry['score'], float) and math.isnan(entry['score']))
}

print(user_ratings)

{'502_Dragon Ball Movie 1: Shen Long no Densetsu': 9, '891_Dragon Ball Movie 2: Majinjou no Nemurihime': 9, '892_Dragon Ball Movie 3: Makafushigi Daibouken': 9, '223_Dragon Ball': 10, '894_Dragon Ball Z Movie 01: Ora no Gohan wo Kaese!!': 10, '895_Dragon Ball Z Movie 02: Kono Yo de Ichiban Tsuyoi Yatsu': 10, '896_Dragon Ball Z Movie 03: Chikyuu Marugoto Choukessen': 10, '897_Dragon Ball Z Movie 04: Super Saiyajin da Son Gokuu': 10, '898_Dragon Ball Z Movie 05: Tobikkiri no Saikyou tai Saikyou': 10, '6714_Dragon Ball Z: Atsumare! Gokuu World': 10, '899_Dragon Ball Z Movie 06: Gekitotsu!! 100-oku Power no Senshi-tachi': 10, '900_Dragon Ball Z Movie 07: Kyokugen Battle!! Sandai Super Saiyajin': 10, '901_Dragon Ball Z Movie 08: Moetsukiro!! Nessen, Ressen, Chougekisen': 10, '902_Dragon Ball Z Movie 09: Ginga Girigiri!! Bucchigiri no Sugoi Yatsu': 10, '984_Dragon Ball Z: Saiya-jin Zetsumetsu Keikaku': 10, '903_Dragon Ball Z Movie 10: Kiken na Futari! Super Senshi wa Nemurenai': 10, '904_Dra

In [8]:
new_user_name = "Current_User"

new_user_row = pd.Series(0, index=anime_names, name=new_user_name)

for anime, rating in user_ratings.items():
    if anime in new_user_row.index:
        new_user_row[anime] = rating


# df_extended = pd.concat([df, new_user_row.to_frame().T])

In [9]:
n = 5
k = 10
max_amount_per_page = 25
delay = 1

In [10]:
def get_recommendations(user, df_norm, anime_names, n=5, k=10):
    
    user_nonzero = user[user != 0]
    user_mean = user_nonzero.mean() if len(user_nonzero) > 0 else 0.0

    new_user_norm = user - user_mean if user_mean != 0 else user.copy()
    new_user_dense = svd.transform(new_user_norm.values.reshape(1, -1)) # Shape: (1, 50)

    knn = NearestNeighbors(n_neighbors=k, metric='cosine')
    knn.fit(dense_user_vectors)

    distances, indices = knn.kneighbors(new_user_dense, n_neighbors=k)
    
    similarities = 1 - distances.flatten()
    neighbor_indices = indices.flatten()
    

    similar_users = pd.Series(similarities[1:], index=neighbor_indices[1:])    

    neighbor_ratings = pd.DataFrame(
        df_norm[similar_users.index].toarray(),
        index=similar_users.index,
        columns=anime_names
    )
    candidate_mask = (neighbor_ratings > 0).any(axis=0) & (user == 0)
    candidate_anime = neighbor_ratings.columns[candidate_mask]
    
    predictions = {}
    for anime in candidate_anime:
        id_ref, name = anime.split('_', 1)
        if int(id_ref) in not_wanted_set:
            continue        
        
        relevant_indices = similar_users.index[neighbor_ratings[anime] > 0]
        
        if len(relevant_indices) > 0:
            weights = similar_users.loc[relevant_indices]
            norm_ratings = neighbor_ratings.loc[relevant_indices, anime]
            
            if weights.sum() > 0: pass
            pred_deviation = np.average(norm_ratings, weights=weights)
            predictions[anime] = round(user_mean + pred_deviation, 2)
    
    if not predictions:
        return {}
    recommendations = pd.Series(predictions).sort_values(ascending=False)
    return recommendations.head(n).to_dict()

In [11]:
recommendations = get_recommendations(new_user_row, df_normalized, anime_names, n, k)

In [12]:
predicted = {}
if len(recommendations) < n:
    amount = (n - len(recommendations) + 1) // max_amount_per_page
    for page in range(1, amount + 2):
        url = f"https://api.jikan.moe/v4/top/anime"
        params = {
            "page": page,
        }
        
        response = requests.get(url, params=params)
        
        data = response.json();
        animes = data['data']

        for anime in data['data']:
            if len(recommendations) < n:
                recommendations[f"{str(anime['mal_id'])}_{anime['title']}"] = 0

In [13]:
for anime_key, score in recommendations.items():
    id_ref, name = anime_key.split('_', 1)
    print(f"Recommend: {name} (ID: {id_ref}) with predicted score: {score}")

Recommend: Douluo Dalu II: Jueshi Tangmen (ID: 51836) with predicted score: 12.29
Recommend: Quanzhi Gaoshou Specials (ID: 37078) with predicted score: 12.29
Recommend: Tunshi Xingkong 4th Season (ID: 56524) with predicted score: 12.29
Recommend: Black Clover: Mahou Tei no Ken (ID: 48585) with predicted score: 12.06
Recommend: Innocence (ID: 468) with predicted score: 12.0


In [14]:
print(time.time() - t0)

0.8708212375640869
